# Gaussian Mixture Regime Allocation

Tests a mixture-model regime classification and corresponding portfolio rules.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
GM-REGIME STRATEGY - COMPLETE INTEGRATED ANALYSIS
==================================================
Single execution cell combining backtesting, walk-forward validation,
robustness metrics, and comprehensive visualization suite.

Run this cell to generate complete analysis output.
"""

from _market_data import download_price_records
import pandas as pd
import numpy as np
from arch import arch_model
from scipy.special import softmax
from scipy.optimize import differential_evolution
from scipy.stats import norm, skew, kurtosis, probplot
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
import seaborn as sns
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")


class Config:
    TRADE_SYMBOL = "SPY"
    BENCHMARK_SYMBOL = "URTH"
    START_DATE = "2013-12-31"
    END_DATE = "2021-12-31"
    WARMUP_DAYS = 252
    TRAIN_DAYS = 252
    VALIDATE_DAYS = 126
    TEST_DAYS = 126
    INITIAL_CAPITAL = 100000.0
    COMPOUNDING_MODE = True
    LEVERAGE = 1.0
    COMMISSION_PER_SHARE = 0.005
    COMMISSION_MIN = 1.0
    COMMISSION_MAX = 0.005
    SLIPPAGE_BPS = 5.0
    GARCH_LOW_P = 1
    GARCH_LOW_Q = 1
    GARCH_HIGH_P = 2
    GARCH_HIGH_Q = 2
    LOW_VOL_THRESHOLD = 0.3
    HIGH_VOL_THRESHOLD = 0.7
    LOW_VOL_ALLOCATION = 1.0
    TRANSITION_ALLOCATION = 0.5
    HIGH_VOL_ALLOCATION = 0.0
    TRADE_BUFFER = 0.05
    REGIME_PIVOT = 0.01
    REGIME_SCALE = 100.0
    GARCH_MIN_WINDOW = 100
    MIN_GRI = 0.65


def get_fmp_data(symbol: str, start: str, end: str) -> pd.DataFrame:
    response_prices = {"historical": download_price_records(symbol, start, end)}
    data = response_prices
    df = pd.DataFrame(data["historical"])
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date")
    df = df.rename(columns={"adjClose": "adj_close"})
    return df[["open", "close", "adj_close", "volume"]]


def compute_log_returns(prices: pd.Series) -> pd.Series:
    return np.log(prices / prices.shift(1)).dropna()


def fit_garch_component(returns: pd.Series, p: int, q: int, rescale: float = 100.0) -> np.ndarray:
    """
    Fit GARCH with EXPANDING WINDOW to prevent look-ahead bias.

    CRITICAL FIX: Instead of fitting once on entire series (which uses future data),
    we refit GARCH parameters at each step using only past data.

    Flow:
        - For day T: Use data up to T-1 to estimate parameters
        - Generate volatility forecast for T
        - Move to T+1

    This is computationally expensive but essential for valid backtesting.
    """
    try:
        sigma_estimates = np.full(len(returns), np.nan)
        scaled_returns = returns * rescale
        min_window = max(Config.GARCH_MIN_WINDOW, p + q + 50)
        for i in range(len(returns)):
            if i < min_window:
                if i > 20:
                    sigma_estimates[i] = scaled_returns.iloc[:i].tail(20).std() / rescale
                else:
                    sigma_estimates[i] = np.nan
                continue
            historical_returns = scaled_returns.iloc[:i]
            try:
                model = arch_model(
                    historical_returns, vol="GARCH", p=p, q=q, dist="t", rescale=False
                )
                result = model.fit(disp="off", options={"maxiter": 100}, show_warning=False)
                sigma_estimates[i] = result.conditional_volatility.iloc[-1] / rescale
            except:
                sigma_estimates[i] = historical_returns.tail(21).std() / rescale
        return sigma_estimates
    except Exception as e:
        print(f"GARCH expanding window failed: {e}")
        return returns.rolling(21, min_periods=21).std().fillna(returns.std()).values


def compute_regime_probabilities(
    sigma_low: np.ndarray, sigma_high: np.ndarray, a1: float = -3.0, a2: float = 3.0
) -> pd.DataFrame:
    """
    Convert volatility to regime probabilities using Config parameters.
    Uses Config.REGIME_PIVOT and Config.REGIME_SCALE instead of hardcoded values.
    """
    avg_sigma = (sigma_low + sigma_high) / 2.0
    centered_input = (avg_sigma - Config.REGIME_PIVOT) * Config.REGIME_SCALE
    logits = np.vstack([a1 * centered_input, a2 * centered_input])
    probs = softmax(logits, axis=0)
    return pd.DataFrame({"p_low": probs[0], "p_high": probs[1]})


def generate_regime_signals(
    returns: pd.Series,
    garch_low_p: int,
    garch_low_q: int,
    garch_high_p: int,
    garch_high_q: int,
    low_threshold: float,
    high_threshold: float,
    a1: float = -3.0,
    a2: float = 3.0,
) -> pd.DataFrame:
    """Generate regime signals with configurable a1/a2 coefficients"""
    sigma_low = fit_garch_component(returns, garch_low_p, garch_low_q)
    sigma_high = fit_garch_component(returns, garch_high_p, garch_high_q)
    probs = compute_regime_probabilities(sigma_low, sigma_high, a1, a2)
    probs.index = returns.index
    regime = np.full(len(probs), 1)
    regime[probs["p_high"] < low_threshold] = 0
    regime[probs["p_high"] >= high_threshold] = 2
    probs["regime"] = regime
    allocation_map = {
        0: Config.LOW_VOL_ALLOCATION,
        1: Config.TRANSITION_ALLOCATION,
        2: Config.HIGH_VOL_ALLOCATION,
    }
    probs["target_allocation"] = probs["regime"].map(allocation_map)
    return probs


def calculate_commission(shares: float, price: float) -> float:
    if shares == 0:
        return 0.0
    base_commission = abs(shares) * Config.COMMISSION_PER_SHARE
    trade_value = abs(shares * price)
    max_commission = trade_value * Config.COMMISSION_MAX
    return max(Config.COMMISSION_MIN, min(base_commission, max_commission))


def calculate_slippage(shares: float, price: float) -> float:
    return abs(shares * price) * (Config.SLIPPAGE_BPS / 10000.0)


def calculate_total_costs(shares: float, price: float) -> float:
    return calculate_commission(shares, price) + calculate_slippage(shares, price)


class BacktestEngine:

    def __init__(self, price_data: pd.DataFrame, signals: pd.DataFrame):
        self.price_data = price_data
        self.signals = signals.shift(1)
        self.initial_capital = Config.INITIAL_CAPITAL
        self.cash = Config.INITIAL_CAPITAL
        self.position_shares = 0.0
        self.portfolio_value = Config.INITIAL_CAPITAL
        self.last_open_portfolio = Config.INITIAL_CAPITAL
        self.equity_curve = []

    def run(self) -> pd.DataFrame:
        dates = self.price_data.index
        for i, date in enumerate(dates):
            if i == 0:
                self.equity_curve.append(
                    {"date": date, "portfolio_value": self.portfolio_value, "daily_return": 0.0}
                )
                continue
            prev_date = dates[i - 1]
            open_t = self.price_data.loc[date, "open"]
            close_t = self.price_data.loc[date, "adj_close"]
            close_prev = self.price_data.loc[prev_date, "adj_close"]
            prev_portfolio_close = self.cash + self.position_shares * close_prev
            portfolio_at_open_today = self.cash + self.position_shares * open_t
            traded = False
            has_signal = date in self.signals.index
            if has_signal and (not pd.isna(self.signals.loc[date, "target_allocation"])):
                target_alloc = self.signals.loc[date, "target_allocation"]
                pos_val = self.position_shares * close_prev
                current_alloc = pos_val / prev_portfolio_close if prev_portfolio_close > 0 else 0
                if abs(target_alloc - current_alloc) > Config.TRADE_BUFFER:
                    if Config.COMPOUNDING_MODE:
                        sizing_base = prev_portfolio_close
                    else:
                        sizing_base = self.initial_capital
                    target_notional = sizing_base * target_alloc * Config.LEVERAGE
                    target_shares = target_notional / close_prev if close_prev > 0 else 0
                    shares_diff = target_shares - self.position_shares
                    if abs(shares_diff) > 0.1:
                        trade_val = shares_diff * open_t
                        costs = calculate_total_costs(shares_diff, open_t)
                        self.cash -= trade_val + costs
                        self.position_shares = target_shares
                        traded = True
            if traded:
                curr_portfolio_open = self.cash + self.position_shares * open_t
                daily_return = (
                    curr_portfolio_open / self.last_open_portfolio - 1.0
                    if self.last_open_portfolio > 0
                    else 0.0
                )
            else:
                curr_portfolio_close = self.cash + self.position_shares * close_t
                daily_return = (
                    curr_portfolio_close / prev_portfolio_close - 1.0
                    if prev_portfolio_close > 0
                    else 0.0
                )
            self.last_open_portfolio = (
                portfolio_at_open_today if not traded else self.cash + self.position_shares * open_t
            )
            self.portfolio_value = self.cash + self.position_shares * close_t
            self.equity_curve.append(
                {
                    "date": date,
                    "portfolio_value": self.portfolio_value,
                    "daily_return": daily_return,
                }
            )
        return pd.DataFrame(self.equity_curve).set_index("date")


def calculate_performance_metrics(returns: pd.Series, label: str = "Strategy") -> Dict:
    returns = returns.dropna()
    if len(returns) == 0:
        return {}
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252.0
    annual_return = (1 + total_return) ** (1 / years) - 1 if years > 0 else 0.0
    annual_vol = returns.std() * np.sqrt(252)
    sharpe = annual_return / annual_vol if annual_vol > 0 else 0.0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0.0
    win_rate = (returns > 0).sum() / len(returns)
    return {
        "label": label,
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "win_rate": win_rate,
        "skewness": skew(returns),
        "kurtosis": kurtosis(returns),
        "num_periods": len(returns),
    }


def optimize_parameters_in_sample(train_data: pd.DataFrame, validate_data: pd.DataFrame) -> Dict:
    """
    Optimize parameters using proper train/validate split.

    NOTE: Currently train_data is not used for pre-fitting (e.g. GARCH calibration).
    All optimization happens on validate_data only. This is acceptable since GARCH
    fitting happens within each backtest using expanding windows.

    For stricter separation, could pre-fit GARCH on train_data and use those
    parameters as priors in validate_data fitting.

    FIT on train_data (currently: N/A), EVALUATE on validate_data.
    """

    def objective(params):
        low_thresh, high_thresh, a1, a2 = params
        if low_thresh >= high_thresh:
            return 1000000.0
        val_returns = compute_log_returns(validate_data["adj_close"])
        try:
            signals = generate_regime_signals(
                val_returns,
                Config.GARCH_LOW_P,
                Config.GARCH_LOW_Q,
                Config.GARCH_HIGH_P,
                Config.GARCH_HIGH_Q,
                low_thresh,
                high_thresh,
                a1,
                a2,
            )
            engine = BacktestEngine(validate_data, signals)
            results = engine.run()
            val_sharpe = (
                results["daily_return"].mean() / results["daily_return"].std() * np.sqrt(252)
            )
            return -val_sharpe
        except:
            return 1000000.0

    bounds = [(0.2, 0.4), (0.6, 0.8), (-5.0, -1.0), (1.0, 5.0)]
    result = differential_evolution(
        objective, bounds=bounds, maxiter=20, seed=42, workers=1, polish=False
    )
    return {
        "low_threshold": result.x[0],
        "high_threshold": result.x[1],
        "a1": result.x[2],
        "a2": result.x[3],
        "val_sharpe": -result.fun,
    }


def walk_forward_analysis(price_data: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    train_days = Config.TRAIN_DAYS
    val_days = Config.VALIDATE_DAYS
    test_days = Config.TEST_DAYS
    warmup = Config.WARMUP_DAYS
    dates = price_data.index
    total_days = len(dates)
    min_required = warmup + train_days + val_days + test_days
    if total_days < min_required:
        raise ValueError(f"Insufficient data: need {min_required} days, have {total_days}")
    fold_results = []
    all_test_returns = []
    all_test_dates = []
    fold_num = 0
    current_start = warmup
    while current_start + train_days + val_days + test_days <= total_days:
        fold_num += 1
        train_start = 0
        train_end = current_start + train_days
        val_end = train_end + val_days
        test_end = val_end + test_days
        train_dates = dates[train_start:train_end]
        val_dates = dates[train_end:val_end]
        test_dates = dates[val_end:test_end]
        print(
            f"\nFold {fold_num}: Train {train_dates[0].date()} to {train_dates[-1].date()}, Val {val_dates[0].date()} to {val_dates[-1].date()}, Test {test_dates[0].date()} to {test_dates[-1].date()}"
        )
        train_data = price_data.loc[train_dates]
        val_data = price_data.loc[val_dates]
        test_data = price_data.loc[test_dates]
        optimal_params = optimize_parameters_in_sample(train_data, val_data)
        print(
            f"  Optimal: low={optimal_params['low_threshold']:.3f}, high={optimal_params['high_threshold']:.3f}, a1={optimal_params['a1']:.3f}, a2={optimal_params['a2']:.3f}"
        )
        test_returns = compute_log_returns(test_data["adj_close"])
        test_signals = generate_regime_signals(
            test_returns,
            Config.GARCH_LOW_P,
            Config.GARCH_LOW_Q,
            Config.GARCH_HIGH_P,
            Config.GARCH_HIGH_Q,
            optimal_params["low_threshold"],
            optimal_params["high_threshold"],
            optimal_params["a1"],
            optimal_params["a2"],
        )
        engine = BacktestEngine(test_data, test_signals)
        test_results = engine.run()
        test_metrics = calculate_performance_metrics(
            test_results["daily_return"], label=f"Fold {fold_num}"
        )
        fold_results.append(
            {
                "fold": fold_num,
                "test_start": test_dates[0],
                "test_end": test_dates[-1],
                "is_sharpe": optimal_params["val_sharpe"],
                "oos_sharpe": test_metrics["sharpe_ratio"],
                "oos_return": test_metrics["annual_return"],
                "oos_vol": test_metrics["annual_volatility"],
                "max_dd": test_metrics["max_drawdown"],
                "params": optimal_params,
            }
        )
        all_test_returns.extend(test_results["daily_return"].tolist())
        all_test_dates.extend(test_results.index.tolist())
        current_start += test_days
    wf_results = pd.DataFrame(fold_results)
    wf_results["wfe"] = wf_results.apply(
        lambda row: row["oos_sharpe"] / row["is_sharpe"] if row["is_sharpe"] != 0 else 0.0, axis=1
    )
    aggregated_returns = pd.Series(all_test_returns, index=all_test_dates)
    return (wf_results, aggregated_returns)


def calculate_deflated_sharpe(
    sharpe_obs: float,
    num_returns: int,
    num_trials: int = 1,
    skewness: float = 0.0,
    kurtosis: float = 3.0,
) -> float:
    gamma = 0.5772
    expected_max_sr = np.sqrt(2 * np.log(num_trials)) - (gamma + np.log(np.log(num_trials))) / (
        2 * np.sqrt(2 * np.log(num_trials))
    )
    var_sharpe = (1 + (1 - skewness * sharpe_obs + (kurtosis - 1) / 4 * sharpe_obs**2)) / (
        num_returns - 1
    )
    dsr = (sharpe_obs - expected_max_sr) / np.sqrt(var_sharpe)
    return norm.cdf(dsr)


def calculate_hurst_exponent(returns: pd.Series, max_lag: int = 20) -> float:
    returns = returns.dropna()
    if len(returns) < max_lag * 2:
        return 0.5
    lags = range(2, max_lag)
    tau = []
    for lag in lags:
        n_chunks = len(returns) // lag
        rs_values = []
        for i in range(n_chunks):
            chunk = returns.iloc[i * lag : (i + 1) * lag].values
            mean_chunk = chunk.mean()
            deviations = chunk - mean_chunk
            cumsum = np.cumsum(deviations)
            R = cumsum.max() - cumsum.min()
            S = chunk.std()
            if S > 0:
                rs_values.append(R / S)
        if len(rs_values) > 0:
            tau.append(np.mean(rs_values))
    if len(tau) > 0:
        log_lags = np.log(list(lags)[: len(tau)])
        log_tau = np.log(tau)
        return np.polyfit(log_lags, log_tau, 1)[0]
    return 0.5


def calculate_rolling_ic(
    signals_df: pd.DataFrame, returns_df: pd.Series, window: int = 60
) -> float:
    """
    Calculate Rolling Information Coefficient.
    Measures correlation between target allocation and forward 1-day returns.
    """
    common_idx = signals_df.index.intersection(returns_df.index)
    if len(common_idx) < window:
        return 0.15
    allocations = signals_df.loc[common_idx, "target_allocation"]
    forward_returns = returns_df.loc[common_idx].shift(-1)
    ic_series = allocations.rolling(window).corr(forward_returns)
    ic_mean = ic_series.dropna().mean()
    return ic_mean if not np.isnan(ic_mean) else 0.15


def calculate_conditional_drawdown_risk(returns: pd.Series, alpha: float = 0.05) -> float:
    """Calculate Conditional Drawdown at Risk (CDaR)."""
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdowns = (cumulative - running_max) / running_max
    threshold = drawdowns.quantile(alpha)
    tail_drawdowns = drawdowns[drawdowns <= threshold]
    return tail_drawdowns.mean() if len(tail_drawdowns) > 0 else 0.0
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdowns = (cumulative - running_max) / running_max
    threshold = drawdowns.quantile(alpha)
    tail_drawdowns = drawdowns[drawdowns <= threshold]
    return tail_drawdowns.mean() if len(tail_drawdowns) > 0 else 0.0


def calculate_global_robustness_index(
    dsr: float, wfe: float, hurst: float, cdar: float, psi: float, ic_mean: float
) -> float:
    dsr_norm = max(0.0, min(1.0, dsr))
    wfe_norm = max(0.0, min(1.0, (wfe - 0.5) / 1.0 + 0.5))
    hurst_norm = 1.0 - 2 * abs(hurst - 0.5)
    cdar_norm = max(0.0, min(1.0, 1.0 + cdar / 0.3))
    psi_norm = psi
    ic_norm = max(0.0, min(1.0, (ic_mean + 1) / 2))
    weights = {"dsr": 0.25, "wfe": 0.2, "hurst": 0.1, "cdar": 0.15, "psi": 0.15, "ic": 0.15}
    gri = (
        weights["dsr"] * dsr_norm
        + weights["wfe"] * wfe_norm
        + weights["hurst"] * hurst_norm
        + weights["cdar"] * cdar_norm
        + weights["psi"] * psi_norm
        + weights["ic"] * ic_norm
    )
    return gri


def plot_equity_curves(strategy_results: pd.DataFrame, benchmark_results: pd.DataFrame):
    fig = plt.figure(figsize=(14, 10))
    gs = GridSpec(3, 1, height_ratios=[2, 1, 1], hspace=0.3)
    ax1 = fig.add_subplot(gs[0])
    strategy_norm = (
        strategy_results["portfolio_value"] / strategy_results["portfolio_value"].iloc[0] * 100
    )
    benchmark_norm = benchmark_results["adj_close"] / benchmark_results["adj_close"].iloc[0] * 100
    ax1.plot(
        strategy_norm.index, strategy_norm, label="GM-Regime Strategy", linewidth=2, color="#2E86AB"
    )
    ax1.plot(
        benchmark_norm.index,
        benchmark_norm,
        label="Benchmark (URTH)",
        linewidth=2,
        color="#A23B72",
        alpha=0.7,
    )
    ax1.set_title("GM-Regime Strategy vs Benchmark", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Portfolio Value (Base 100)", fontsize=11)
    ax1.legend(loc="best", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    cumulative = strategy_norm / 100
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max * 100
    ax2.fill_between(drawdown.index, drawdown, 0, alpha=0.3, color="red")
    ax2.plot(drawdown.index, drawdown, color="darkred", linewidth=1)
    ax2.set_ylabel("Drawdown (%)", fontsize=11)
    ax2.set_title("Strategy Drawdown", fontsize=11, fontweight="bold")
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    bench_cumulative = benchmark_norm / 100
    bench_running_max = bench_cumulative.expanding().max()
    bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max * 100
    ax3.fill_between(bench_drawdown.index, bench_drawdown, 0, alpha=0.3, color="purple")
    ax3.plot(bench_drawdown.index, bench_drawdown, color="darkviolet", linewidth=1)
    ax3.set_ylabel("Drawdown (%)", fontsize=11)
    ax3.set_xlabel("Date", fontsize=11)
    ax3.set_title("Benchmark Drawdown", fontsize=11, fontweight="bold")
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
    ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    return fig


def plot_regime_timeline(
    signals: pd.DataFrame, prices: pd.DataFrame, low_thresh: float = 0.3, high_thresh: float = 0.7
):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(prices.index, prices["adj_close"], color="black", linewidth=1.5, label="SPY Price")
    regime_colors = {0: "#90EE90", 1: "#FFD700", 2: "#FF6B6B"}
    regime_labels = {0: "Low-Vol", 1: "Transition", 2: "High-Vol"}
    for regime_id, color in regime_colors.items():
        mask = signals["regime"] == regime_id
        ax1.fill_between(
            signals.index,
            prices["adj_close"].min(),
            prices["adj_close"].max(),
            where=mask,
            alpha=0.2,
            color=color,
            label=regime_labels[regime_id],
        )
    ax1.set_ylabel("SPY Price ($)", fontsize=11)
    ax1.set_title("Market Regime Detection Timeline", fontsize=14, fontweight="bold")
    ax1.legend(loc="upper left", fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax2.plot(
        signals.index, signals["p_high"], color="#C44569", linewidth=2, label="High-Vol Probability"
    )
    ax2.axhline(
        y=low_thresh,
        color="green",
        linestyle="--",
        linewidth=1,
        alpha=0.7,
        label=f"Low-Vol Threshold ({low_thresh:.2f})",
    )
    ax2.axhline(
        y=high_thresh,
        color="red",
        linestyle="--",
        linewidth=1,
        alpha=0.7,
        label=f"High-Vol Threshold ({high_thresh:.2f})",
    )
    ax2.set_ylabel("P(High-Vol)", fontsize=11)
    ax2.set_xlabel("Date", fontsize=11)
    ax2.set_ylim(0, 1)
    ax2.legend(loc="upper right", fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    return fig


def plot_walk_forward_results(wf_results: pd.DataFrame):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ax1 = axes[0, 0]
    x = wf_results["fold"]
    width = 0.35
    ax1.bar(
        x - width / 2, wf_results["is_sharpe"], width, label="In-Sample", color="skyblue", alpha=0.8
    )
    ax1.bar(
        x + width / 2,
        wf_results["oos_sharpe"],
        width,
        label="Out-of-Sample",
        color="salmon",
        alpha=0.8,
    )
    ax1.set_xlabel("Fold", fontsize=11)
    ax1.set_ylabel("Sharpe Ratio", fontsize=11)
    ax1.set_title("Sharpe Ratio: In-Sample vs Out-of-Sample", fontsize=12, fontweight="bold")
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis="y")
    ax2 = axes[0, 1]
    colors = [
        "green" if wfe >= 0.7 else "orange" if wfe >= 0.5 else "red" for wfe in wf_results["wfe"]
    ]
    ax2.bar(wf_results["fold"], wf_results["wfe"], color=colors, alpha=0.7)
    ax2.axhline(y=1.0, color="blue", linestyle="--", linewidth=2, label="Perfect WFE")
    ax2.axhline(y=0.7, color="green", linestyle="--", linewidth=1, alpha=0.7)
    ax2.set_xlabel("Fold", fontsize=11)
    ax2.set_ylabel("WFE Ratio", fontsize=11)
    ax2.set_title("Walk-Forward Efficiency by Fold", fontsize=12, fontweight="bold")
    ax2.legend(loc="best", fontsize=9)
    ax2.grid(True, alpha=0.3, axis="y")
    ax3 = axes[1, 0]
    ax3.bar(wf_results["fold"], wf_results["max_dd"] * 100, color="darkred", alpha=0.7)
    ax3.set_xlabel("Fold", fontsize=11)
    ax3.set_ylabel("Max Drawdown (%)", fontsize=11)
    ax3.set_title("Maximum Drawdown by Fold", fontsize=12, fontweight="bold")
    ax3.grid(True, alpha=0.3, axis="y")
    ax4 = axes[1, 1]
    colors_return = ["green" if ret > 0 else "red" for ret in wf_results["oos_return"]]
    ax4.bar(wf_results["fold"], wf_results["oos_return"] * 100, color=colors_return, alpha=0.7)
    ax4.set_xlabel("Fold", fontsize=11)
    ax4.set_ylabel("Annual Return (%)", fontsize=11)
    ax4.set_title("Out-of-Sample Annual Return by Fold", fontsize=12, fontweight="bold")
    ax4.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    return fig


def plot_robustness_metrics(metrics: Dict[str, float]):
    fig, ax = plt.subplots(figsize=(12, 8))
    thresholds = {
        "Deflated Sharpe Ratio": 0.7,
        "Walk-Forward Efficiency": 0.7,
        "Hurst Exponent": 0.5,
        "Conditional Drawdown at Risk (5%)": -0.15,
        "Parameter Stability Index": 0.7,
        "Global Robustness Index": 0.65,
    }
    labels = list(metrics.keys())
    values = list(metrics.values())
    threshold_vals = [thresholds.get(label, 0.5) for label in labels]
    colors = []
    for label, value in zip(labels, values):
        thresh = thresholds.get(label, 0.5)
        if "Drawdown" in label:
            colors.append("green" if value >= thresh else "red")
        elif "Hurst" in label:
            colors.append("green" if abs(value - 0.5) < 0.15 else "orange")
        else:
            colors.append("green" if value >= thresh else "red")
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, values, color=colors, alpha=0.7, edgecolor="black")
    for i, thresh in enumerate(threshold_vals):
        ax.plot([thresh, thresh], [i - 0.4, i + 0.4], "k--", linewidth=2, alpha=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Metric Value", fontsize=11)
    ax.set_title("Strategy Robustness Assessment", fontsize=14, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="x")
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(
            val + 0.02,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}",
            va="center",
            fontsize=9,
            fontweight="bold",
        )
    plt.tight_layout()
    return fig


print("=" * 70)
print("GM-REGIME STRATEGY - COMPLETE INTEGRATED ANALYSIS")
print("=" * 70)
print("\n[1/5] Downloading data...")
start_dt = datetime.strptime(Config.START_DATE, "%Y-%m-%d")
extended_start = (start_dt - timedelta(days=Config.WARMUP_DAYS + 100)).strftime("%Y-%m-%d")
trade_data = get_fmp_data(Config.TRADE_SYMBOL, extended_start, Config.END_DATE)
bench_data = get_fmp_data(Config.BENCHMARK_SYMBOL, extended_start, Config.END_DATE)
print(
    f"✓ {Config.TRADE_SYMBOL}: {len(trade_data)} days | {Config.BENCHMARK_SYMBOL}: {len(bench_data)} days"
)
print("\n[2/5] Running walk-forward validation...")
wf_results, aggregated_returns = walk_forward_analysis(trade_data)
print(f"✓ Completed {len(wf_results)} folds")
print("\n[3/5] Calculating performance metrics...")
strat_metrics = calculate_performance_metrics(aggregated_returns, label="GM-Regime")
bench_returns = compute_log_returns(bench_data["adj_close"])
common_dates = aggregated_returns.index.intersection(bench_returns.index)
bench_returns_aligned = bench_returns.loc[common_dates]
bench_metrics = calculate_performance_metrics(bench_returns_aligned, label=Config.BENCHMARK_SYMBOL)
print("\n[4/5] Calculating robustness metrics...")
dsr = calculate_deflated_sharpe(
    strat_metrics["sharpe_ratio"],
    len(aggregated_returns),
    len(wf_results),
    strat_metrics["skewness"],
    strat_metrics["kurtosis"],
)
avg_wfe = wf_results["wfe"].mean()
hurst = calculate_hurst_exponent(aggregated_returns)
cdar = calculate_conditional_drawdown_risk(aggregated_returns, alpha=0.05)
best_fold_sharpe = wf_results["oos_sharpe"].max()
worst_fold_sharpe = wf_results["oos_sharpe"].min()
psi = max(0.0, worst_fold_sharpe / best_fold_sharpe) if best_fold_sharpe > 0 else 0.0
test_trade_data = trade_data.loc[aggregated_returns.index]
test_price_returns = compute_log_returns(test_trade_data["adj_close"])
median_low = wf_results.apply(lambda r: r["params"]["low_threshold"], axis=1).median()
median_high = wf_results.apply(lambda r: r["params"]["high_threshold"], axis=1).median()
median_a1 = wf_results.apply(lambda r: r["params"]["a1"], axis=1).median()
median_a2 = wf_results.apply(lambda r: r["params"]["a2"], axis=1).median()
test_signals = generate_regime_signals(
    test_price_returns,
    Config.GARCH_LOW_P,
    Config.GARCH_LOW_Q,
    Config.GARCH_HIGH_P,
    Config.GARCH_HIGH_Q,
    median_low,
    median_high,
    median_a1,
    median_a2,
)
ic_mean = calculate_rolling_ic(test_signals, aggregated_returns, window=60)
gri = calculate_global_robustness_index(dsr, avg_wfe, hurst, cdar, psi, ic_mean)
robustness_metrics = {
    "Deflated Sharpe Ratio": dsr,
    "Walk-Forward Efficiency": avg_wfe,
    "Hurst Exponent": hurst,
    "Conditional Drawdown at Risk (5%)": cdar,
    "Parameter Stability Index": psi,
    "Global Robustness Index": gri,
}
print("\n[5/5] Generating visualizations...")
median_low_thresh = wf_results.apply(lambda row: row["params"]["low_threshold"], axis=1).median()
median_high_thresh = wf_results.apply(lambda row: row["params"]["high_threshold"], axis=1).median()
median_a1 = wf_results.apply(lambda row: row["params"]["a1"], axis=1).median()
median_a2 = wf_results.apply(lambda row: row["params"]["a2"], axis=1).median()
full_returns = compute_log_returns(trade_data["adj_close"])
full_signals = generate_regime_signals(
    full_returns,
    Config.GARCH_LOW_P,
    Config.GARCH_LOW_Q,
    Config.GARCH_HIGH_P,
    Config.GARCH_HIGH_Q,
    median_low_thresh,
    median_high_thresh,
    median_a1,
    median_a2,
)
full_engine = BacktestEngine(trade_data, full_signals)
full_results = full_engine.run()
print("\nPlot 1: Equity Curves")
fig1 = plot_equity_curves(full_results, bench_data)
plt.show()
plt.savefig(
    "results/gaussian-mixture-regime-allocation/01_equity_curves.png", dpi=300, bbox_inches="tight"
)
plt.close()
print("Plot 2: Regime Timeline")
fig2 = plot_regime_timeline(full_signals, trade_data, median_low_thresh, median_high_thresh)
plt.show()
plt.savefig(
    "results/gaussian-mixture-regime-allocation/02_regime_timeline.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("Plot 3: Walk-Forward Analysis")
fig3 = plot_walk_forward_results(wf_results)
plt.show()
plt.savefig(
    "results/gaussian-mixture-regime-allocation/03_walk_forward_analysis.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
print("Plot 4: Robustness Metrics")
fig4 = plot_robustness_metrics(robustness_metrics)
plt.show()
plt.savefig(
    "results/gaussian-mixture-regime-allocation/04_robustness_metrics.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
wf_results.drop("params", axis=1).to_csv(
    "results/gaussian-mixture-regime-allocation/walk_forward_results.csv", index=False
)
full_results.to_csv("results/gaussian-mixture-regime-allocation/equity_curve.csv")
full_signals.to_csv("results/gaussian-mixture-regime-allocation/regime_signals.csv")
print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"\n{'Metric':<35} {'Strategy':>12} {'Benchmark':>12}")
print("-" * 70)
print(
    f"{'Annual Return':<35} {strat_metrics['annual_return'] * 100:>11.2f}% {bench_metrics['annual_return'] * 100:>11.2f}%"
)
print(
    f"{'Annual Volatility':<35} {strat_metrics['annual_volatility'] * 100:>11.2f}% {bench_metrics['annual_volatility'] * 100:>11.2f}%"
)
print(
    f"{'Sharpe Ratio':<35} {strat_metrics['sharpe_ratio']:>12.3f} {bench_metrics['sharpe_ratio']:>12.3f}"
)
print(
    f"{'Max Drawdown':<35} {strat_metrics['max_drawdown'] * 100:>11.2f}% {bench_metrics['max_drawdown'] * 100:>11.2f}%"
)
print(
    f"{'Calmar Ratio':<35} {strat_metrics['calmar_ratio']:>12.3f} {bench_metrics['calmar_ratio']:>12.3f}"
)
print("\n" + "=" * 70)
print("ROBUSTNESS METRICS")
print("=" * 70)
for key, value in robustness_metrics.items():
    print(f"{key:<40} {value:>8.4f}")
production_ready = gri >= Config.MIN_GRI
print("\n" + "=" * 70)
print("PRODUCTION READINESS ASSESSMENT")
print("=" * 70)
print(f"\nGlobal Robustness Index: {gri:.4f}")
print(f"Threshold for Production: {Config.MIN_GRI:.4f}")
print(
    f"\nStatus: {('✓ APPROVED FOR PRODUCTION' if production_ready else '✗ NOT READY FOR PRODUCTION')}"
)
print("\n" + "=" * 70)
print("✓ Analysis complete. All outputs saved to /mnt/user-data/outputs/")
print("=" * 70)
